# Multi-zone CUDA fault — focused reproducer

On an A100 (Colab, torch 2.11+cu128, Python 3.13) every multi-zone single-shooting case died with
`CUDA error: an illegal memory access was encountered`; 1-zone cases passed, and the same 10-zone cases pass on
the laptop (RTX PRO 2000, torch 2.13+cu130, Python 3.12).

**Status (2026-09-08).** Probe runs on the A100 established: the capture routine (eager warmup, graph
record, one replay, replay-vs-eager parity check) completes with clean per-phase device syncs, the first
bundle call returns correct values (objective 100.0), and the **next replay of the same graph with identical
inputs faults**. The memory sampler, the inputs, and the recording are cleared. The graph references device
memory that is released between the two replays. Remaining hypotheses, and the switch that tests each:

| Hypothesis | Switch | Expected if true |
|---|---|---|
| expandable-segment allocator defers recycling until capture ends, then unmaps pages the graph uses | `ALLOC_CONF = "expandable_segments:False"` | passes (cell 2 prints whether segments are expandable) |
| torch 2.11 CUDA-graph bug (fixed by 2.13) | `TORCH_SPEC = "torch==2.13.0 --index-url https://download.pytorch.org/whl/cu128"` | passes |
| fault needs graph capture/replay | `BACKEND = "eager"` | passes |

`SYNC_PHASES` sets `T4B_CUDA_GRAPH_SYNC_PHASES=1`: the wrapper synchronizes after every capture phase so the
asynchronous fault is attributed to the phase that launched it, and it adds a note to the exception.

`CUDA_LAUNCH_BLOCKING` makes kernel launches synchronous. That is **incompatible with stream capture** (the
record phase will fail with a different error), so use it only together with `BACKEND = "eager"` if eager
also faults, to name the kernel.

* `MODE` `smoke` (2 h horizon, 1 iteration, minutes) vs `full` (120 h, 300 iterations); `MAXITER` caps the
  iterations independently of the mode (0 = mode default). The crashing case is `full` / 10 zones / `cuda_graph`.
* `derivative_stats` in the output shows `captures` and `replays`; a run with `replays: 0` never executed the
  captured graph outside the capture routine's own parity replay.

Run the cells top to bottom. Cell 1 must run **before** anything imports torch. Restart the runtime after a
crash (the CUDA context is poisoned) and after changing `TORCH_SPEC`.

In [ ]:
#@title 1. Configuration (runs before torch is imported) { display-mode: "form" }
import os
REPOSITORY = "https://github.com/JBjoernskov/Twin4Build.git"  #@param {type:"string"}
GIT_REF = "feature/issue-128/collocation-initialization"       #@param {type:"string"}
N_ZONES = 10                    #@param {type:"integer"}
SOLVER = "slsqp-single-shooting"  #@param ["slsqp-single-shooting", "custom-batched-sqp", "ipopt-collocation"]
N_STARTS = 1                    #@param {type:"integer"}
BACKEND = "cuda_graph"          #@param ["cuda_graph", "eager"]
MODE = "smoke"                  #@param ["smoke", "full"]
MAXITER = 0                     #@param {type:"integer"}
#@markdown `MAXITER` 0 = the mode's default (smoke: 1, full: 300). Use `MODE="full"` with `MAXITER=3` to
#@markdown capture the real 120 h / 10-zone graph and replay it a few times without waiting for 300 iterations.
MEMORY_SAMPLER = "fixed"        #@param ["off", "fixed", "legacy_mem_get_info"]
SYNC_PHASES = True              #@param {type:"boolean"}
PROBE = True                    #@param {type:"boolean"}
#@markdown `PROBE` replaces the SLSQP solve by a fixed sequence of bundle calls (capture at x0, replay at x0,
#@markdown replay at perturbed points) with a device sync after each, so the first faulting call is named.
CUDA_LAUNCH_BLOCKING = False    #@param {type:"boolean"}
TORCH_SPEC = ""                 #@param {type:"string"}
ALLOC_CONF = ""                 #@param ["", "expandable_segments:False", "expandable_segments:True"]
#@markdown `ALLOC_CONF` sets `PYTORCH_CUDA_ALLOC_CONF` before torch is imported (empty = keep the runtime's).
#@markdown With expandable segments the allocator defers memory recycling until a capture ends; a graph whose
#@markdown buffers span physical segments can then read unmapped pages on replay. Windows (the laptop that
#@markdown passes) cannot use expandable segments at all.
#@markdown `TORCH_SPEC` empty = keep the runtime's torch. Example to match the laptop that passes:
#@markdown `torch==2.13.0 --index-url https://download.pytorch.org/whl/cu128` (restart the runtime after changing it).
os.environ["CUDA_LAUNCH_BLOCKING"] = "1" if CUDA_LAUNCH_BLOCKING else "0"
os.environ["T4B_CUDA_GRAPH_SYNC_PHASES"] = "1" if SYNC_PHASES else "0"
if ALLOC_CONF:
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ALLOC_CONF
print("PYTORCH_CUDA_ALLOC_CONF (before torch import):", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
os.environ["T4B_BENCHMARK_MODE"] = MODE
print({k: v for k, v in globals().items() if k in ("N_ZONES", "SOLVER", "N_STARTS", "BACKEND", "MODE", "MAXITER", "MEMORY_SAMPLER", "SYNC_PHASES", "PROBE", "CUDA_LAUNCH_BLOCKING", "TORCH_SPEC", "ALLOC_CONF")})

In [ ]:
#@title 2. Install Twin4Build from the branch (same bootstrap as the benchmark notebooks)
import pathlib, subprocess, sys
REPO = pathlib.Path("/content/Twin4Build")
if "google.colab" in sys.modules:
    if not REPO.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=False, capture_output=True)
    if TORCH_SPEC.strip():
        print("installing", TORCH_SPEC)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *TORCH_SPEC.split()], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "psutil"], check=True)
else:
    REPO = pathlib.Path.cwd()
    if REPO.name == "benchmarks":
        REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("repo at", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
import torch, twin4build
from twin4build.utils import _cuda_graph
assert hasattr(_cuda_graph, "mark_phase"), "stale checkout: phase markers missing - delete /content/Twin4Build and rerun"
print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("CUDA_LAUNCH_BLOCKING =", os.environ.get("CUDA_LAUNCH_BLOCKING"), "T4B_CUDA_GRAPH_SYNC_PHASES =", os.environ.get("T4B_CUDA_GRAPH_SYNC_PHASES"))
# Allocator diagnostics: backend, env config, and whether segments are expandable (from a live snapshot).
_probe_tensor = torch.ones(1024, device="cuda")
_segments = torch.cuda.memory_snapshot()
print("allocator backend:", torch.cuda.get_allocator_backend(), "| PYTORCH_CUDA_ALLOC_CONF:", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"),
      "| expandable segments:", sorted({bool(seg.get("is_expandable", False)) for seg in _segments}) if _segments else "no segments")
del _probe_tensor

In [ ]:
#@title 3. Build the case exactly as the harness does
import time, json, traceback, threading
import benchmarks.common as common
from benchmarks.common import BenchmarkConfig, batched_estimation_problem, seed_everything, _estimation_window, ESTIMATION_METHODS
import twin4build as tb

config = BenchmarkConfig(mode=MODE)
seed_everything(config.seed)
t0 = time.time()
setup = batched_estimation_problem(N_ZONES, config)
model = setup["model"]; model.to("cuda", torch.float64)
print(f"setup built in {time.time()-t0:.0f} s: {N_ZONES} zones, hours={config.hours}, maxiter={config.estimation_maxiter}")

# Optional: re-create the OLD sampler (CUDA runtime call from a side thread) to reproduce deliberately.
legacy_stop = threading.Event()
def _legacy_poll():
    while not legacy_stop.wait(1.0):
        try:
            torch.cuda.mem_get_info()
        except Exception as exc:
            print("legacy sampler error:", repr(exc)[:200])
if MEMORY_SAMPLER == "legacy_mem_get_info":
    threading.Thread(target=_legacy_poll, daemon=True, name="legacy-mem-sampler").start()
    print("legacy mem_get_info sampler thread started (1 Hz)")
elif MEMORY_SAMPLER == "off":
    common.MEMORY_SAMPLER_INTERVAL_SECONDS = 3600.0  # effectively disables the fixed sampler

In [ ]:
#@title 4. Run the case in-process (full traceback on failure)
estimator = tb.Estimator(tb.Simulator(model, execution_mode="functional", execution_backend=BACKEND))
options = {"maxiter": MAXITER or config.estimation_maxiter}
method = ESTIMATION_METHODS[SOLVER]
if SOLVER == "ipopt-collocation":
    options["hessian"] = "exact"
if SOLVER == "custom-batched-sqp":
    options.update({"n_starts": N_STARTS, "batch_size": N_STARTS, "start_seed": config.seed, "start_strategy": "local", "start_spread": 0.1})
print("method", method, "options", options, "backend", BACKEND, "probe", PROBE)

class ProbeDone(Exception):
    pass

PROBE_LOG = []
def _probe_solve(self, method, n_cores, options):
    # Replaces the solver: a fixed sequence of bundle calls, device-synced, first fault named.
    import numpy as np
    import twin4build.utils.types as tps
    from twin4build.estimator._batched_solvers import BatchedObjectiveEvaluator
    ev = BatchedObjectiveEvaluator(self._functional_objective)
    x0 = torch.as_tensor(np.asarray(self._x0_norm, dtype=np.float64), dtype=tps.float_dtype(), device=self._device).unsqueeze(0)
    lb = torch.as_tensor(self.bounds.lb, dtype=x0.dtype, device=x0.device)
    ub = torch.as_tensor(self.bounds.ub, dtype=x0.dtype, device=x0.device)
    gen = torch.Generator(device="cpu").manual_seed(0)
    steps = [
        ("capture@x0", x0),
        ("replay@x0 (identical inputs)", x0.clone()),
        ("replay@x0+1e-9 (tiny perturbation)", x0 + 1e-9),
        ("replay@x0+1e-3*rand (small step)", torch.clamp(x0 + 1e-3 * torch.rand(x0.shape, generator=gen).to(x0), lb, ub)),
        ("replay@x0+0.1*rand (SLSQP-sized step)", torch.clamp(x0 + 0.1 * torch.rand(x0.shape, generator=gen).to(x0), lb, ub)),
        ("replay@x0 again", x0.clone()),
    ]
    ref = None
    for label, x in steps:
        t = time.time()
        try:
            v, g = ev.value_grad(x)
            torch.cuda.synchronize()
            row = {"step": label, "ok": True, "value": float(v[0]), "grad_abs_max": float(g.abs().max()), "seconds": round(time.time() - t, 1)}
            if ref is None:
                ref = (v.clone(), g.clone())
            elif label.startswith("replay@x0"):
                row["matches_capture"] = bool(torch.allclose(v, ref[0]) and torch.allclose(g, ref[1]))
        except Exception as exc:
            row = {"step": label, "ok": False, "error": repr(exc)[:200], "last_phase": _cuda_graph.LAST_PHASE, "seconds": round(time.time() - t, 1)}
            PROBE_LOG.append(row); print(row)
            raise
        PROBE_LOG.append(row); print(row)
    raise ProbeDone()

if PROBE:
    tb.Estimator._dispatch_solve = _probe_solve  # runs after the functional objective is set up
torch.cuda.reset_peak_memory_stats()
t0 = time.time(); outcome = None
try:
    result, seconds = common.timed("cuda", lambda: estimator.estimate(
        parameters=setup["parameters"], measurements=setup["measurements"], method=method, options=options, **_estimation_window(config)))
    outcome = {"status": "ok", "seconds": seconds, "last_phase": _cuda_graph.LAST_PHASE, "torch": torch.__version__, "backend": BACKEND, "iterations": result.get("iterations"), "message": result.get("message"),
               "final_objective": result.get("final_objective"), "derivative_stats": result.get("derivative_stats")}
except ProbeDone:
    outcome = {"status": "probe complete, no fault", "torch": torch.__version__, "backend": BACKEND,
               "alloc_conf": os.environ.get("PYTORCH_CUDA_ALLOC_CONF"), "probe": PROBE_LOG}
except BaseException as exc:
    chain = [exc, exc.__cause__, exc.__context__]
    notes = [n for e in chain if e is not None for n in getattr(e, "__notes__", [])]
    outcome = {"status": "FAILED", "seconds": time.time() - t0, "error": repr(exc)[:400], "capture_phase_notes": notes,
               "last_phase": _cuda_graph.LAST_PHASE, "torch": torch.__version__, "backend": BACKEND,
               "alloc_conf": os.environ.get("PYTORCH_CUDA_ALLOC_CONF"), "probe": PROBE_LOG}
    traceback.print_exc()
finally:
    legacy_stop.set()
print(json.dumps(outcome, indent=1, default=str))
print("memory:", json.dumps({k: (round(v / 2**30, 2) if isinstance(v, int) and v > 1e6 else v) for k, v in common.memory_stats_of_last_timed().items()}, indent=1))
print("torch peak allocated GiB", round(torch.cuda.max_memory_allocated() / 2**30, 2), "reserved", round(torch.cuda.max_memory_reserved() / 2**30, 2))

### How to read the outcome

**Probe runs** (`PROBE=True`, the default) print one row per bundle call. The first row with `"ok": false`
names the call that faulted:

| First failing step | Meaning |
|---|---|
| `capture@x0` | the capture routine itself (warmup / record / internal replay); see `last_phase` |
| `replay@x0 (identical inputs)` | any replay outside the capture routine faults, inputs irrelevant: graph or its buffers invalid after the routine returns |
| `replay@x0+1e-9` or later | replay is fine with identical inputs and faults once inputs change: data-dependent indexing/branching inside the captured rollout |
| none, but `matches_capture: false` | replays run but return stale/garbage values |
| none (with `BACKEND="eager"`) | eager evaluation at the same points is fine: the fault needs graph replay |

Suggested order, restarting the runtime between runs (all `MODE="full"`, 10 zones, `PROBE=True`):

1. `cuda_graph`, `ALLOC_CONF = "expandable_segments:False"` — cell 2 must print `expandable segments: [False]`.
2. `cuda_graph`, `TORCH_SPEC = "torch==2.13.0 --index-url https://download.pytorch.org/whl/cu128"`.
3. `eager` — same points evaluated eagerly.

With `PROBE=False` the cell runs the real SLSQP solve; `last_phase` then says which bundle step the fault
surfaced in (`capture:*` inside the routine, `value_grad:*` in the caller). Paste the outcome JSON back.

In [ ]:
#@title 5. (Optional) the same case through the real harness subprocess, with the fixed sampler
RUN_HARNESS = False  #@param {type:"boolean"}
if RUN_HARNESS:
    from benchmarks.common import _run_estimation_case_subprocess
    row = _run_estimation_case_subprocess(config, n_zones=N_ZONES, device="cuda", solver=SOLVER, n_starts=N_STARTS, repetition=0)
    print({k: row.get(k) for k in ("status", "seconds", "iterations", "message", "error", "child_returncode")})
    if row.get("child_traceback"):
        print(row["child_traceback"][-3000:])